# Task 2: Laning & Overtaking with SB3 PPO

This notebook trains a PPO agent on the newer `highway-env` / Stable-Baselines3 API, following the older Task 2 setup from `racetrack-agents` where three slower non-agent vehicles are spawned and the ego vehicle must lane-follow while overtaking.

The older DQN command used `--spawn_vehicles 3`, `--batch_size 256`, `--lr 0.00005`, `--lr_decay`, `--arch Identity`, and `--fc_layers 3`. The cells below map those ideas to SB3 PPO with a 3-layer MLP policy, linear learning-rate decay, and `other_vehicles=3` in the `racetrack-oval-v0` config.

In [ ]:
# If this notebook is running in a fresh environment, install the core packages first.
# In the local repo environment you can usually leave this cell commented out.
#
# %pip install "highway-env>=1.8" "stable-baselines3[extra]>=2.0" tensorboard moviepy

from pathlib import Path
from copy import deepcopy
import base64
import os
import random

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import highway_env  # Registers highway-env environments in many versions.
import numpy as np
import torch.nn as nn
import torch

from IPython.display import HTML, display
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecEnv

# Newer gymnasium versions can register an external environment package explicitly.
# Older highway-env versions register on import, so we keep this tolerant.
try:
    gym.register_envs(highway_env)
except Exception:
    pass


## Experiment Config

Task 2 is represented by `other_vehicles=3`. The notebook now starts from `RacetrackEnvOval.default_config()` and overrides only the pieces that define this experiment, so future highway-env API changes are easier to absorb.

In [ ]:
SEED = 42
ENV_ID = "racetrack-v0"

# Keep full training as the default. For an end-to-end notebook smoke test, run
# `FAST_DEV_RUN=1` in the process environment before executing the notebook.
FAST_DEV_RUN = True
USE_EARLY_STOP = False  # Stop training early if the evaluation reward is consistently high.

# Prefer CPU in notebooks unless a CUDA-enabled GPU is available and stable.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

EXP_ID = "sb3_ppo_custom_2phase_wall_fric"
if FAST_DEV_RUN:
    EXP_ID += "_fastdev"

WORK_DIR = Path.cwd()

RUN_DIR = WORK_DIR / "runs" / EXP_ID
MODEL_DIR = RUN_DIR / "models"
BEST_MODEL_DIR = MODEL_DIR / "best"
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
LOG_DIR = RUN_DIR / "logs"
VIDEO_DIR = RUN_DIR / "videos"
TB_LOG_DIR = RUN_DIR / "tensorboard"

for directory in [MODEL_DIR, BEST_MODEL_DIR, CHECKPOINT_DIR, LOG_DIR, VIDEO_DIR, TB_LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Reproducibility: exact runs can still vary across machines/GPU kernels.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
def set_global_seeds(seed: int, fast_training: bool = True):
    """
    Seed every random source that affects training.
    Call this ONCE before creating envs or model.
    """
    random.seed(seed)                          # Python stdlib
    np.random.seed(seed)                       # NumPy
    torch.manual_seed(seed)                    # PyTorch CPU ops
    torch.cuda.manual_seed_all(seed)           # PyTorch GPU ops (all devices)
    os.environ["PYTHONHASHSEED"] = str(seed)   # Python hash randomisation

    # Optional: sacrifice speed for full CUDA determinism.
    # Comment out if training speed matters more than exact reproducibility.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False  # benchmark=True auto-tunes kernels
                                                 # but picks different ops each run
    if fast_training:
        # Let cuDNN find the fastest algorithm for your fixed input shapes.
        # One-time profiling cost of ~2-3 seconds at the start of training.
        # Not reproducible: two runs may produce slightly different loss curves
        # but converge to the same policy quality.
        torch.backends.cudnn.benchmark     = True
        torch.backends.cudnn.deterministic = False
    else:
        # Force deterministic algorithms throughout.
        # Reproducible: two runs with the same seed produce byte-identical results.
        # Costs 5-30% training speed depending on your CNN architecture.
        torch.backends.cudnn.benchmark     = False
        torch.backends.cudnn.deterministic = True

set_global_seeds(SEED, fast_training=not FAST_DEV_RUN)

In [ ]:
from highway_env.envs import racetrack_env
from race_env import RacetrackFast
racetrack_env.RacetrackFast = RacetrackFast
gym.register(id=ENV_ID, entry_point="race_env:RacetrackFast")

In [ ]:
N_ENVS = min(8, max(1, os.cpu_count() or 1))

# ════════════════════════════════════════════════════════════════════
# Speed and step count optimisation
# ════════════════════════════════════════════════════════════════════

if FAST_DEV_RUN:
    PHASE1_TIMESTEPS = 65_536    
    PHASE2_TIMESTEPS = 8_192
    P1_N_STEPS    = 512;  P2_N_STEPS    = 512
    P1_BATCH_SIZE =  128;  P2_BATCH_SIZE =  128
    P1_N_EPOCHS   =   8;  P2_N_EPOCHS   =   8
    P1_ENT_COEF   = 0.10; P2_ENT_COEF   = 0.01
    P1_LR         = 8e-4; P2_LR         = 2e-4
    P1_EVAL_FREQ  =  P1_N_STEPS * 4; P2_EVAL_FREQ  =  P1_N_STEPS * 4
    N_EVAL_EPISODES = 1;  CHECKPOINT_FREQ = 4096

else:
    # Reduced steps — sufficient with dense rewards + fast sim
    PHASE1_TIMESTEPS = 1_000_000   # was 2_000_000
    PHASE2_TIMESTEPS = 3_000_000   # was 6_000_000

    # Shorter rollout = more frequent updates per wall-clock minute
    P1_N_STEPS    = 512
    P1_BATCH_SIZE = 256
    P1_N_EPOCHS   =  15
    P1_ENT_COEF   = 0.02
    P1_LR         = 2e-4    # slightly higher → faster convergence in phase 1

    P2_N_STEPS    = 1024    # was 2048 — more frequent updates
    P2_BATCH_SIZE = 256
    P2_N_EPOCHS   =  15
    P2_ENT_COEF   = 0.005
    P2_LR         = 1e-4

    # eval_freq = exact multiple of n_steps
    P1_EVAL_FREQ  = P1_N_STEPS * 8    # =  4096 — every 8 rollouts
    P2_EVAL_FREQ  = P2_N_STEPS * 8    # =  8192

    N_EVAL_EPISODES = 5
    CHECKPOINT_FREQ = P2_N_STEPS * 20  # every 20 rollouts

# Slightly smaller clip range helps learning stay stable when the action space grows.
CLIP_RANGE = 0.15

# Stronger discounting and GAE smoothing can improve horizon-aware lane-following and overtaking behavior.
GAMMA = 0.99
GAE_LAMBDA = 0.98
MAX_GRAD_NORM = 0.5


## Build Training and Evaluation Environments

SB3 trains on vectorized environments. `DummyVecEnv` is the safest default inside notebooks on Windows. For longer command-line runs, set `USE_SUBPROC = True`.

In [ ]:
class EnvFactory:
    """
    Picklable factory with explicit render_mode control.
    render_mode is always None during training.
    Pass render_mode="rgb_array" only for recording envs.
    """
    def __init__(self, config: dict, render_mode: str | None = None):
        self.config      = config
        self.render_mode = render_mode   # None during training, always

    def __call__(self) -> gym.Env:
        return gym.make(
            ENV_ID,
            config      = self.config,
            render_mode = self.render_mode,  # explicitly passed, never hardcoded
        )

In [ ]:
# Phase 1: no NPCs, learn basic forward motion
phase1_config = RacetrackFast.default_config().copy()
phase1_config["other_vehicles"] = 0

# Phase 2: add NPCs for avoidance
phase2_config = RacetrackFast.default_config().copy()
phase2_config["other_vehicles"] = 1

# Eval: no NPCs (cleaner signal), phase 2 config otherwise
eval_config = RacetrackFast.default_config().copy()
eval_config["other_vehicles"] = 0
eval_config["terminate_off_road"]= True

In [ ]:
def make_fresh_eval_env(seed: int) -> VecEnv:
    """
    Always creates a brand-new VecEnv for evaluation.
    Never reuse an eval_env across phases — stale episode state
    causes EvalCallback to get inconsistent rewards.
    """
    return make_vec_env(
        EnvFactory(eval_config, render_mode=None),
        n_envs      = 1,
        seed        = seed,
        vec_env_cls = DummyVecEnv,
    )

In [ ]:
phase1_train_env = make_vec_env(
    EnvFactory(phase1_config, render_mode=None),
    n_envs      = N_ENVS,
    seed        = SEED,
    vec_env_cls = DummyVecEnv,
)

phase2_train_env = make_vec_env(
    EnvFactory(phase2_config, render_mode=None),
    n_envs      = N_ENVS,
    seed        = SEED,
    vec_env_cls = DummyVecEnv,
)

## Define PPO

The PPO policy uses a 3-layer actor and critic MLP, matching the spirit of `--arch Identity --fc_layers 3` from the older code: flatten the occupancy grid, then learn dense policy/value heads. Comments below explain every model setting that differs from SB3 defaults or maps to the older command.

In [ ]:
class RacetrackCNN(BaseFeaturesExtractor):
    """
    Lightweight CNN for the 11×12×12 OccupancyGrid.
    Input:  [batch, 11, 12, 12]
    Output: flat feature vector of size features_dim
    """
    def __init__(self, observation_space, features_dim=512):
        super().__init__(observation_space, features_dim)
        n_channels = observation_space.shape[0]  # 11
        self.cnn = nn.Sequential(
            nn.Conv2d(n_channels, 64, kernel_size=3, padding=1),  # → [64, 12, 12]
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),          # → [128, 12, 12]
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=3, stride=2),            # → [256, 5, 5]
            nn.ReLU(),
            nn.Flatten(),                                           # → 1600
            nn.Linear(256 * 5 * 5, features_dim),
            nn.ReLU(),
        )

    def forward(self, obs):
        return self.cnn(obs.float())

policy_kwargs = dict(
    features_extractor_class=RacetrackCNN,
    features_extractor_kwargs=dict(features_dim=512),
    net_arch=dict(pi=[256, 128], vf=[256, 128]),  # separate actor/critic heads
)

In [ ]:
def linear_schedule(initial_value):
    """SB3 schedule: progress_remaining moves from 1.0 to 0.0 during training."""
    def schedule(progress_remaining):
        return progress_remaining * initial_value
    return schedule

In [ ]:
phase1_model = PPO(
    policy          = "CnnPolicy",
    env             = phase1_train_env,
    learning_rate   = linear_schedule(P1_LR),
    n_steps         = P1_N_STEPS,
    batch_size      = P1_BATCH_SIZE,
    n_epochs        = P1_N_EPOCHS,
    gamma           = GAMMA,
    gae_lambda      = GAE_LAMBDA,
    clip_range      = CLIP_RANGE,
    ent_coef        = P1_ENT_COEF,
    max_grad_norm   = MAX_GRAD_NORM,
    policy_kwargs   = policy_kwargs,
    tensorboard_log = str(TB_LOG_DIR),
    seed            = SEED,
    verbose         = 1,
    device          = DEVICE,
)

# SB3 uses orthogonal init with scale 0.01 — mean output is near 0
# Manually shift the throttle bias toward +0.5 so initial acceleration
# maps to lmap(0.5, [-1,1], [-3,6]) = 3.75 m/s² instead of 1.5 m/s²
with torch.no_grad():
    # phase1_model.policy.action_net.bias.zero_()
    
    action_bias = phase1_model.policy.action_net.bias
    action_bias[0] = 0.15 if FAST_DEV_RUN else 0.40   # throttle dimension → biased forward
    action_bias[1] = 0.0    # steering dimension → neutral

## Train and Save the Best Model

`EvalCallback` periodically runs deterministic evaluations and writes the best model to disk. TensorBoard logs are stored under `runs/sb3_ppo_task2_laning_overtaking/logs`.

In [ ]:
# Start TensorBoard in the notebook while training is running.
# This opens the log directory in a background server and prints the local URL.
#
%load_ext tensorboard
%tensorboard --logdir TB_LOG_DIR --port 6006

# If the magic above is not available, you can also launch a standalone server from a terminal:
# tensorboard --logdir "runs/sb3_ppo_laning_overtaking_richocc_throttle_fastdev/tensorboard" --host 127.0.0.1 --port 6006


In [ ]:
class PhaseEarlyStop(BaseCallback):
    """
    Stop a training phase when eval reward has not improved
    for `patience` consecutive evaluations.
    Reads EvalCallback's logged value — no per-step signal.
    """
    def __init__(self, patience: int = 5, min_delta: float = 0.5, verbose: int = 1):
        super().__init__(verbose)
        self.patience   = patience
        self.min_delta  = min_delta
        self._best      = -np.inf
        self._no_improv = 0
        self._last_val  = None

    def _on_step(self) -> bool:
        val = self.logger.name_to_value.get("eval/mean_reward", None)
        if val is None or val == self._last_val:
            return True
        self._last_val = val

        if val > self._best + self.min_delta:
            self._best      = val
            self._no_improv = 0
            if self.verbose:
                print(f"[EarlyStop] Improved → {self._best:.3f}")
        else:
            self._no_improv += 1
            if self.verbose:
                print(f"[EarlyStop] No improvement {self._no_improv}/{self.patience} "
                      f"(current={val:.3f} best={self._best:.3f})")

        if self._no_improv >= self.patience:
            if self.verbose:
                print(f"[EarlyStop] Stopping at step {self.num_timesteps}")
            return False
        return True

In [ ]:
def make_callbacks(phase_label: str, n_steps: int, eval_freq: int):
    fresh_eval_env = make_vec_env(
        EnvFactory(eval_config, render_mode=None),
        n_envs=1, seed=SEED + 10_000, vec_env_cls=DummyVecEnv,
    )
    save_path = BEST_MODEL_DIR / phase_label
    save_path.mkdir(parents=True, exist_ok=True)

    eval_cb = EvalCallback(
        fresh_eval_env,
        best_model_save_path = str(save_path),
        log_path             = str(LOG_DIR / "eval" / phase_label),
        eval_freq            = eval_freq,
        n_eval_episodes      = N_EVAL_EPISODES,
        deterministic        = True,
        render               = False,
        verbose              = 1,
    )
    ckpt_cb = CheckpointCallback(
        save_freq   = CHECKPOINT_FREQ,
        save_path   = str(CHECKPOINT_DIR / phase_label),
        name_prefix = f"ppo_{phase_label}",
        verbose     = 0,
    )
    callbacks = [eval_cb, ckpt_cb]

    if USE_EARLY_STOP and not FAST_DEV_RUN:
        # Phase 1: stop quickly once driving is learned (patience=5)
        # Phase 2: allow more exploration before giving up (patience=8)
        patience = 5 if phase_label == "phase1" else 8
        callbacks.append(
            PhaseEarlyStop(patience=patience, min_delta=0.5, verbose=1)
        )

    return callbacks, fresh_eval_env

In [ ]:
phase1_callbacks, phase1_eval_env = make_callbacks(
    "phase1", P1_N_STEPS, P1_EVAL_FREQ
)

In [ ]:
phase1_model.learn(
    total_timesteps = PHASE1_TIMESTEPS,
    callback        = phase1_callbacks,
    tb_log_name     = "phase1",
    progress_bar    = True,
)

In [ ]:
phase1_path = MODEL_DIR / "phase1.zip"
phase1_model.save(str(phase1_path))
print(f"Phase 1 saved → {phase1_path}")
phase1_eval_env.close()

In [ ]:
best_p1 = BEST_MODEL_DIR / "phase1" / "best_model.zip"
print(f"\nPhase 1 best model exists: {best_p1.exists()} → {best_p1}")
assert best_p1.exists(), (
    "Phase 1 best_model.zip was NOT saved. "
    "Check that EvalCallback printed eval results above. "
    f"Expected path: {best_p1}"
)

In [ ]:
phase2_callbacks, phase2_eval_env = make_callbacks(
    "phase2", P2_N_STEPS, P2_EVAL_FREQ
)

In [ ]:
model = PPO.load(
    str(phase1_path),
    env            = phase2_train_env,
    device         = DEVICE,

    # custom_objects overrides saved hyperparameters and
    # triggers a full _setup_model() reinitialization
    custom_objects = {
        "learning_rate": linear_schedule(P2_LR),   # lower LR for fine-tuning
        "n_steps":       P2_N_STEPS,               # ← buffer reallocated here
        "batch_size":    P2_BATCH_SIZE,
        "n_epochs":      P2_N_EPOCHS,
        "ent_coef":      P2_ENT_COEF,              # lower entropy: exploit more
        "clip_range":    CLIP_RANGE,
        "gamma":         GAMMA,
        "gae_lambda":    GAE_LAMBDA,
        "max_grad_norm": MAX_GRAD_NORM,
    },
)

In [ ]:
# Verify buffer was correctly reallocated
assert model.rollout_buffer.buffer_size == P2_N_STEPS, (
    f"Buffer size mismatch: expected {P2_N_STEPS}, "
    f"got {model.rollout_buffer.buffer_size}. "
    f"custom_objects may not have applied correctly."
)
print(f"Buffer size confirmed: {model.rollout_buffer.buffer_size}")

In [ ]:
model.learn(
    total_timesteps     = PHASE2_TIMESTEPS,
    callback            = phase2_callbacks,
    tb_log_name         = "phase2",
    reset_num_timesteps = False,   # continue global step counter
    progress_bar        = True,
)

In [ ]:
final_path = MODEL_DIR / "ppo_last.zip"
model.save(str(final_path))
print(f"Phase 2 saved → {final_path}")
phase2_eval_env.close()

In [ ]:
import importlib

for module_name in ["onnx", "onnxruntime"]:
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        os.system('pip install --quiet "onnx==1.12.0" "onnxruntime==1.12.0"')

import onnx
import onnxruntime as ort

In [ ]:
best_model_path = BEST_MODEL_DIR / "phase2"  / "best_model.zip"
print(f"\nPhase 2 best model exists: {best_model_path.exists()} → {best_model_path}")

last_model_path = MODEL_DIR / "ppo_last.zip"

trained_model = PPO.load(
    best_model_path if best_model_path.exists() else last_model_path,
    device="cpu",
)
trained_model.policy.eval()
trained_model.policy.to("cpu")

obs_shape = phase2_train_env.observation_space.shape      # e.g. (11, 12, 12)
dummy_obs  = torch.zeros(1, *obs_shape, dtype=torch.float32)  # [1,C,H,W]

print(f"Observation space : {obs_shape}")
print(f"Dummy input shape : {tuple(dummy_obs.shape)}")

In [ ]:
class ActorOnlyWrapper(nn.Module):
    """
    obs → action_mean
    Deterministic forward pass: features → mlp_pi → action_net.
    No sampling, no value head.  Use this in RE Engine.
    """
    def __init__(self, policy):
        super().__init__()
        self.policy = policy

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        # obs arrives as [batch, C, H, W] — 4D, no reshaping needed.
        features          = self.policy.features_extractor(obs.float())
        latent_pi, _      = self.policy.mlp_extractor(features)
        return self.policy.action_net(latent_pi)   # [batch, 2]

In [ ]:
class FullPolicyWrapper(nn.Module):
    """
    obs → (action_mean, value)
    Exports both actor and critic heads.
    Useful for debugging / distillation; not needed for RE Engine inference.
    """
    def __init__(self, policy):
        super().__init__()
        self.policy = policy

    def forward(self, obs: torch.Tensor):
        features               = self.policy.features_extractor(obs.float())
        latent_pi, latent_vf   = self.policy.mlp_extractor(features)
        action_mean            = self.policy.action_net(latent_pi)   # [batch, 2]
        value                  = self.policy.value_net(latent_vf)    # [batch, 1]
        return action_mean, value

In [ ]:
def export_and_verify(
    wrapper:      nn.Module,
    dummy_input:  torch.Tensor,
    out_path:     Path,
    output_names: list[str],
    opset:        int = 17,
) -> ort.InferenceSession:

    wrapper.eval()
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Sanity-check forward pass before tracing
    with torch.no_grad():
        test_out = wrapper(dummy_input)
    if isinstance(test_out, tuple):
        for i, t in enumerate(test_out):
            print(f"  pre-export output[{i}]: {tuple(t.shape)}")
    else:
        print(f"  pre-export output: {tuple(test_out.shape)}")

    # ONNX trace
    with torch.no_grad():
        torch.onnx.export(
            wrapper,
            dummy_input,
            str(out_path),
            export_params       = True,
            opset_version       = opset,
            do_constant_folding = True,
            input_names         = ["obs"],
            output_names        = output_names,
            dynamic_axes        = {
                "obs": {0: "batch_size"},
                **{name: {0: "batch_size"} for name in output_names},
            },
            dynamo = False,
        )

    # ONNX structural check
    onnx_model = onnx.load(str(out_path))
    onnx.checker.check_model(onnx_model)

    # Print graph I/O summary
    print(f"\n✅  {out_path.name}")
    for node in onnx_model.graph.input:
        dims = [d.dim_value if d.dim_value else d.dim_param
                for d in node.type.tensor_type.shape.dim]
        print(f"   input  '{node.name}' : {dims}")
    for node in onnx_model.graph.output:
        dims = [d.dim_value if d.dim_value else d.dim_param
                for d in node.type.tensor_type.shape.dim]
        print(f"   output '{node.name}' : {dims}")

    # OnnxRuntime inference check
    sess = ort.InferenceSession(str(out_path), providers=["CPUExecutionProvider"])
    dummy_np  = dummy_input.numpy()
    ort_outs  = sess.run(None, {"obs": dummy_np})
    for out_meta, arr in zip(sess.get_outputs(), ort_outs):
        print(f"   ORT  '{out_meta.name}' : shape={arr.shape}  "
              f"min={arr.min():.4f}  max={arr.max():.4f}")

    return sess

In [ ]:
ACTOR_PATH = MODEL_DIR / "ppo_actor_only.onnx"
FULL_PATH  = MODEL_DIR / "ppo_full_policy.onnx"

In [ ]:
print("=" * 60)
print("Exporting  actor-only  ONNX …")
actor_session = export_and_verify(
    ActorOnlyWrapper(trained_model.policy),
    dummy_obs,
    ACTOR_PATH,
    output_names=["action_mean"],
)

In [ ]:
print()
print("=" * 60)
print("Exporting  full policy  ONNX …")
full_session = export_and_verify(
    FullPolicyWrapper(trained_model.policy),
    dummy_obs,
    FULL_PATH,
    output_names=["action_mean", "value"],
)

In [ ]:
C, H, W = obs_shape
print(f"""
╔══════════════════════════════════════════════════════════╗
║             ONNX Export Summary — C# Reference           ║
╠══════════════════════════════════════════════════════════╣
║  File (RE Engine)   : ppo_actor_only.onnx                ║
║  Input  "obs"       : [1, {C}, {H}, {W}]  (NCHW)              ║
║  Output "action_mean": [1, 2]                            ║
║     action[0] = throttle ∈ [-1,1] → map to accel m/s²   ║
║     action[1] = steering ∈ [-1,1] → map to angle rad     ║
╠══════════════════════════════════════════════════════════╣
║  File (debug only)  : ppo_full_policy.onnx               ║
║  Input  "obs"       : [1, {C}, {H}, {W}]  (NCHW)              ║
║  Output "action_mean": [1, 2]                            ║
║  Output "value"     : [1, 1]  (critic estimate)          ║
╚══════════════════════════════════════════════════════════╝
""".format(C=C, H=H, W=W))

## Load the Best PPO Checkpoint

If training was interrupted before an evaluation improved, fall back to the last saved model.

In [ ]:
best_model_path = BEST_MODEL_DIR / "best_model.zip"
last_model_path = MODEL_DIR / "ppo_last.zip"

if best_model_path.exists():
    trained_model = PPO.load(best_model_path)
    print(f"Loaded best model: {best_model_path}")
else:
    trained_model = PPO.load(last_model_path)
    print(f"Best model was not found, loaded last model: {last_model_path}")


## Find and Export the Best Evaluation Episode

The first pass evaluates deterministic rollouts over fixed seeds without recording. The best seed is then replayed once with `RecordVideo`, producing a single video for the strongest episode found in this sweep.

In [ ]:
def run_episode(model, config, seed, record=False, name_prefix="ppo_cnn_best_episode"):
    """Run one deterministic episode and optionally record it to VIDEO_DIR."""
    render_mode = "rgb_array" if record else None
    env = gym.make(ENV_ID, config=config, render_mode=render_mode)

    if record:
        env = RecordVideo(
            env,
            video_folder=str(VIDEO_DIR),
            name_prefix=name_prefix,
            episode_trigger=lambda episode_id: episode_id == 0,
        )
    obs, info = env.reset(seed=seed)
    terminated = False
    truncated = False
    total_reward = 0.0
    episode_length = 0

    while not (truncated):
        if not FAST_DEV_RUN and terminated:
            break
        action, _ = model.predict(obs[np.newaxis], deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action[0])
        total_reward += float(reward)
        episode_length += 1
        if record:
            env.render()

    env.close()
    return total_reward, episode_length

In [ ]:
rec_config = RacetrackFast.default_config().copy()
rec_config["other_vehicles"] = 1
if FAST_DEV_RUN:
    # Fast dev: disable termination so the episode runs for a fixed
    # duration regardless of the untrained model going off-road.
    # Purpose: verify the recording pipeline works, not show good driving.
    rec_config["terminate_off_road"] = False   # ← key line
    rec_config["duration"]           = 150     # 30 seconds at 5 Hz
    record_steps = 150
    print("[FastDev] Recording with terminate_off_road=False — "
            "car may go off-track, this is expected for an untrained model.")
else:
    rec_config["terminate_off_road"] = True
    rec_config["duration"]           = 1500    # full episode
    record_steps = 1500

candidate_seeds = list(range(SEED, SEED + (1 if FAST_DEV_RUN else 25)))
episode_scores = []

for seed in candidate_seeds:
    reward, length = run_episode(trained_model, rec_config, seed=seed, record=False)
    episode_scores.append({"seed": seed, "reward": reward, "length": length})

best_episode = max(episode_scores, key=lambda item: item["reward"])
best_episode


In [ ]:
video_prefix = f"ppo_task2_best_seed_{best_episode['seed']}"
recorded_reward, recorded_length = run_episode(
    trained_model,
    rec_config,
    seed=best_episode["seed"],
    record=True,
    name_prefix=video_prefix,
)

video_files = sorted(VIDEO_DIR.glob(f"{video_prefix}*.mp4"), key=lambda path: path.stat().st_mtime)
best_video_path = video_files[-1] if video_files else None

print(f"Recorded reward: {recorded_reward:.3f}")
print(f"Recorded length: {recorded_length}")
print(f"Video path: {best_video_path}")


## Display the Exported Video

In [ ]:
def show_video(video_path, width=720):
    """Embed an exported mp4 directly in the notebook."""
    video_path = Path(video_path)
    video_bytes = video_path.read_bytes()
    encoded = base64.b64encode(video_bytes).decode("ascii")
    display(HTML(f"""
    <video width="{width}" controls>
      <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
    </video>
    """))

if best_video_path is not None:
    show_video(best_video_path)
else:
    print("No video file was found. Check that moviepy/ffmpeg are installed and rerun the recording cell.")


## record video env

In [ ]:
from stable_baselines3.common.vec_env import VecVideoRecorder
def record_video(
    model_path: str | Path,
    video_dir:  str | Path,
    n_episodes: int  = 3,
    video_length: int = 300,   # steps per video, match your episode duration
):
    """
    Load trained model and record n_episodes to mp4.
    Completely separate from training — no interaction with train_env.
    """
    video_dir = Path(video_dir)
    video_dir.mkdir(parents=True, exist_ok=True)

    # Load model — device doesn't matter for inference, cpu is fine
    model = PPO.load(str(model_path), device="cpu")

    # Create a fresh env with rgb_array — NEVER reuse train_env here
    # rgb_array must be set at gym.make time; you cannot switch after creation
    record_env = DummyVecEnv([
        lambda: gym.make(ENV_ID, config=RacetrackFast.default_config(),
                         render_mode="rgb_array")
    ])

    # Wrap with recorder — saves an mp4 every `video_length` steps
    record_env = VecVideoRecorder(
        record_env,
        video_folder = str(video_dir),
        record_video_trigger = lambda step: step == 0,  # start immediately
        video_length = video_length,
        name_prefix  = "racetrack_policy",
    )

    obs = record_env.reset()
    for episode in range(n_episodes):
        done  = False
        total = 0.0
        steps = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, info = record_env.step(action)
            total += float(reward[0])
            steps += 1

        print(f"Episode {episode+1}: {steps} steps, return={total:.2f}")
        obs = record_env.reset()

    record_env.close()
    print(f"Videos saved to {video_dir}")

In [ ]:
def record_video_manual(
    model_path:  str | Path,
    output_path: str | Path,
    n_episodes:  int   = 1,
    fps:         int   = 15,    # match simulation_frequency
    resolution:  tuple = (600, 600),
):
    try:
        import cv2
    except ImportError:
        raise ImportError("pip install opencv-python")

    model  = PPO.load(str(model_path), device="cpu")
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)

    cfg = RacetrackFast.default_config().copy()
    cfg["screen_width"]  = resolution[0]
    cfg["screen_height"] = resolution[1]

    # rgb_array must be at construction time — cannot change later
    env = gym.make(ENV_ID, config=cfg, render_mode="rgb_array")

    # OpenCV VideoWriter — mp4v codec is widely compatible
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = None

    for episode in range(n_episodes):
        obs, _ = env.reset()
        done = truncated = False
        total = 0.0

        while not (done or truncated):
            # Render BEFORE step so frame shows the state the policy saw
            frame = env.render()   # [H, W, 3] uint8 RGB

            if writer is None:
                h, w = frame.shape[:2]
                writer = cv2.VideoWriter(str(output), fourcc, fps, (w, h))

            # OpenCV uses BGR; convert from RGB
            writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

            action, _ = model.predict(obs[np.newaxis], deterministic=True)
            obs, reward, done, truncated, info = env.step(action[0])
            total += float(reward)

        print(f"Episode {episode+1}: return={total:.2f}")

    env.close()
    if writer:
        writer.release()
    print(f"Saved: {output}")

In [ ]:
record_video(
    model_path  = BEST_MODEL_DIR / "phase2" / "best_model.zip",
    video_dir   = str(VIDEO_DIR),
    n_episodes  = 3,
    video_length= 300,
)

## Optional: TensorBoard

Run this cell while training or after training to inspect reward, loss, entropy, KL, and evaluation curves.

In [ ]:
# Uncomment these lines in an interactive notebook session.

